In [1]:
import pandas as pd
import os
import requests

In [2]:
urls = pd.read_csv('../results/combined_job_urls.csv')['job_url'].tolist()

In [3]:
url = urls[0]
url

'https://www.linkedin.com/jobs/view/4239122420/?eBP=CwEAAAGZnA-Juc9wRnvqHoznrywIjCWTzLxZJ2FvcHMAW1UhUgp4MU8TxDhICKCK425LcXCZ6ELyaEW-wT9TBzTPAet0phl8QRTagd58cLRVlzfN4xpEDUEqD5r0CrQln_neuYTRVe3uV8h0ssdFXh4gSrf04639FNYxitJuPJHIQ-Pqs7l_U1_nQO_z8omOKSwHM7UgFCI5erQjnBFNTjuUZZDuqSfte59-iOYEDtBVqQvwxEXPKV1qKQOZTt66BFCIK7Q_6cHVLJhTfvaTLpisPBXY1twCfAsOYaFszDI5hWGDij2qcwpN4XuorY9dQRotV2LTi5KGOWKBfCVOUnG1FpLyuFRtnQrrXitXSDP2826TEyRkCf8DuqytK9RGCCexkXwE1YZ2TWLarmaG9n5IoOz5SOP2gJf9vMfeTzaF9cA3esap9_9K8DXYWNUkJcIOvparqt_RiNAL8nawE8F26NVktL79k9ixCmoIHNQtg3bvdl0QxGJqhavrltFqHxh2MPLL5g&refId=eMCFVpx2KOD%2F58MSxepOnw%3D%3D&trackingId=4VKwxSOmdjChq6W%2FyHRzZQ%3D%3D&trk=flagship3_search_srp_jobs'

In [4]:
header = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
response = requests.get(url, headers=header)

In [5]:
response

<Response [200]>

In [6]:
response.text

'<!DOCTYPE html>\n\n    \n    \n    \n    \n    \n    \n    \n\n    \n    <html lang="en">\n      <head>\n        <meta name="pageKey" content="d_jobs_guest_details">\n          \n    <meta name="robots" content="max-image-preview:large, noarchive">\n      <meta name="bingbot" content="max-image-preview:large">\n  \n<!----><!---->        <meta name="locale" content="en_US">\n<!---->        <meta id="config" data-app-version="2.0.2591" data-call-tree-id="AAZAjD7oQ4XZi5yVy5mRrA==" data-multiproduct-name="jobs-guest-frontend" data-service-name="jobs-guest-frontend" data-browser-id="2ffb085f-68ab-4e5b-862a-98480ac77964" data-enable-page-view-heartbeat-tracking data-page-instance="urn:li:page:d_jobs_guest_details;6/6L8RNGRjWivscqXkq+lQ==" data-disable-jsbeacon-pagekey-suffix="false" data-member-id="0" data-should-use-full-url-in-pve-path="true" data-dna-member-lix-treatment="enabled" data-human-member-lix-treatment="enabled" data-dfp-member-lix-treatment="control" data-sync-apfc-headers-lix

In [7]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')

In [8]:
def extract_information_from_html(html):
    results = {
        'job_title': "",
        'company_name': "",
        'company_location': "",
        'salary_range': "",
        'job_description': "",
        'publish_time': "",
        'num_applicants': "",
        "job_description_meta": {}
    }
    soup = BeautifulSoup(html, 'html.parser')
    
    title = soup.find('h1', class_='top-card-layout__title')
    if title:
        results['job_title'] = title.get_text(strip=True)
    company = soup.find('h4', class_='top-card-layout__second-subline')
    if company:
        # first div is 2span - company name and location, second div is 2 span - publish time and number of applicants
        spans = company.find_all('span')
        if len(spans) >= 2:
            results['company_name'] = spans[0].get_text(strip=True)
            results['company_location'] = spans[1].get_text(strip=True)
            results['publish_time'] = spans[2].get_text(strip=True) if len(spans) > 2 else ""
            results['num_applicants'] = spans[3].get_text(strip=True) if len(spans) > 3 else ""

    compensation = soup.find('section', class_='compensation')
    if compensation:
        salary_range = compensation.find('div', class_='compensation__salary-range')
        if salary_range:
            results['salary_range'] = salary_range.get_text(strip=True)
        else:
            results['salary_range'] = ""

    description = soup.find('section', class_='description')
    if description:
        results['job_description'] = description.get_text(strip=True)
    
        # ul.description__job-criteria-list
        job_criteria = description.find('ul', class_='description__job-criteria-list')
        if job_criteria:
            descriptions = {}
            criteria = job_criteria.find_all('li')
            for criterion in criteria:
                key = criterion.find('h3')
                value = criterion.find('span')
                if key and value:
                    descriptions[key.get_text(strip=True)] = value.get_text(strip=True)
            results['job_description_meta'] = descriptions

    return results

In [9]:
html = response.text
job_info = extract_information_from_html(html)
job_info

{'job_title': 'Imaging Student Intern PRN',
 'company_name': 'Alaska Regional Hospital',
 'company_location': 'Anchorage, AK',
 'salary_range': '',
 'job_description': 'DescriptionIntroductionDo you have the PRN career opportunities as a(an) Imaging Student Intern PRN you want with your current employer? We have an exciting opportunity for you to join Alaska Regional Hospital which is part of the nation\'s leading provider of healthcare services, HCA Healthcare.BenefitsAlaska Regional Hospital, offers a total rewards package that supports the health, life, career and retirement of our colleagues. The available plans and programs include:Comprehensive medical coverage that covers many common services at no cost or for a low copay. Plans include prescription drug and behavioral health coverage as well as telemedicine services and free AirMed medical transportation.Additional options for dental and vision benefits, life and disability coverage, flexible spending accounts, supplemental hea

In [10]:
url = urls[23]
html = requests.get(url, headers=header).text
job_info = extract_information_from_html(html)
job_info

{'job_title': 'BU Lead, Data & Analytics',
 'company_name': 'McGraw Hill',
 'company_location': 'United States',
 'salary_range': 'Base pay range$120,000.00/yr - $150,000.00/yr',
 'job_description': "OverviewBuild the FutureAt McGraw Hill, we are dedicated to delivering digital learning experiences that transform education for learners and educators. Our focus is on creating seamless, impactful products that truly benefit our users while supporting growth and collaboration across teams. We foster a culture that values innovation, teamwork, and a balance between career growth and personal well-being.How can you make an impact?The Business Unit Lead for the Data & Analytics (D&A) leads the development and deployment of business intelligence datasets and reporting dashboards for one of McGraw Hill’s four business units (K12, Higher Education, Professional, and International) or corporate functions (HR, Finance),and plays a key role in setting and executing on the strategy for this develop

In [23]:
import tqdm
from time import sleep

In [15]:
job_infos = []

In [32]:
i = 0
for url in tqdm.tqdm(urls[3109:]):
    try:
        html = requests.get(url, headers=header).text
        job_info = extract_information_from_html(html)
        job_infos.append(job_info)
    except Exception as e:
        print(f"Error processing {url}: {e}")
        sleep(120)
        # retry
        html = requests.get(url, headers=header).text
        job_info = extract_information_from_html(html)
        job_infos.append(job_info)
    i += 1
    if i % 100 == 0:
        print(f"Processed {i} job postings")
        sleep(60)

  1%|          | 99/8026 [01:43<2:13:53,  1.01s/it]

Processed 100 job postings


  2%|▏         | 199/8026 [04:26<2:53:28,  1.33s/it] 

Processed 200 job postings


  3%|▎         | 206/8026 [05:33<6:46:45,  3.12s/it] 

...

 98%|█████████▊| 7899/8026 [3:42:24<02:14,  1.06s/it]  

Processed 7900 job postings


100%|█████████▉| 7999/8026 [3:45:10<00:27,  1.04s/it]

Processed 8000 job postings


100%|██████████| 8026/8026 [3:46:39<00:00,  1.69s/it]


In [33]:
len(job_infos)

11135

In [27]:
job_infos[:10]

[{'job_title': 'Imaging Student Intern PRN',
  'company_name': 'Alaska Regional Hospital',
  'company_location': 'Anchorage, AK',
  'salary_range': '',
  'job_description': 'DescriptionIntroductionDo you have the PRN career opportunities as a(an) Imaging Student Intern PRN you want with your current employer? We have an exciting opportunity for you to join Alaska Regional Hospital which is part of the nation\'s leading provider of healthcare services, HCA Healthcare.BenefitsAlaska Regional Hospital, offers a total rewards package that supports the health, life, career and retirement of our colleagues. The available plans and programs include:Comprehensive medical coverage that covers many common services at no cost or for a low copay. Plans include prescription drug and behavioral health coverage as well as telemedicine services and free AirMed medical transportation.Additional options for dental and vision benefits, life and disability coverage, flexible spending accounts, supplementa

In [34]:
df = pd.DataFrame(job_infos)

df['job_url'] = urls

df.to_csv('../results/detailed_job_infos.csv', index=False)